In [1]:
import os
import sys

os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PYSPARK_PYTHON"]        = sys.executable
import time

In [2]:
from pyspark import SparkContext
from sympy import *
from drudge import *
from gristmill import *
 

In [3]:
import re

# ----------------------------------------------------------------------------
# 1) Start Spark & BCS quasiparticle drudge
# ----------------------------------------------------------------------------
ctx = SparkContext('local[*]', 'bcs_ccsd')
dr  = ReducedBCSDrudge(ctx)
dr.full_simplify = True
print("finished")


26/03/26 20:39:04 WARN Utils: Your hostname, Swarnamoys-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.0.0.87 instead (on interface en0)
26/03/26 20:39:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 20:39:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/26 20:39:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/26 20:39:05 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
                                                                                

finished


In [4]:
P, Pdag, N = dr.names.P, dr.names.Pdag, dr.names.N
p,q,r,s,i,j,k,l = dr.names.A_dumms[:8]
u, v = IndexedBase('u'), IndexedBase('v')
P_i_dag = (u[p]*v[p]+u[p]**2*Pdag[p]
     - u[p]*v[p]*N[p]
     - v[p]**2*P[p])

P_i_dag_i = (u[p+r]*v[p+r]+u[p+r]**2*Pdag[p+r]
     - u[p+r]*v[p+r]*N[p+r]
     - v[p+r]**2*P[p+r])

P_j_dag = (
        u[q]*v[q]
      + u[q]**2 * Pdag[q]
      - u[q]*v[q] * N[q]
      - v[q]**2 * P[q]
    )

P_k_dag = (
        u[r]*v[r]
      + u[r]**2 * Pdag[r]
      - u[r]*v[r] * N[r]
      - v[r]**2 * P[r]
    )
P_i = (
        (u[p])*(v[p])
      + (u[p])**2 * P[p]
      - (u[p])*(v[p]) * N[p]
      - (v[p])**2 * Pdag[p]
    )

P_i_i = (
        (u[p+r])*(v[p+r])
      + (u[p+r])**2 * P[p+r]
      - (u[p+r])*(v[p+r]) * N[p+r]
      - (v[p+r])**2 * Pdag[p+r]
    )

P_j = ((u[q])*(v[q])
      + (u[q])**2 * P[q]
      - (u[q])*(v[q]) * N[q]
      - (v[q])**2 * Pdag[q])

P_k = ((u[r])*(v[r])
      + (u[r])**2 * P[r]
      - (u[r])*(v[r]) * N[r]
      - (v[r])**2 * Pdag[r])
#hc = P_j_dag * P_i
N_i = 2*(v[p])*v[p] +((u[p])*u[p] - (v[p])*v[p])*N[p] +2*u[p]*(v[p])*Pdag[p]\
            +2*(u[p])*v[p]*P[p]
N_i_r = 2*(v[p+r])*v[p+r] +((u[p+r])*u[p+r] - (v[p+r])*v[p+r])*N[p+r] +2*u[p+r]*(v[p+r])*Pdag[p+r]\
            +2*(u[p+r])*v[p+r]*P[p+r]
N_j = 2*(v[q])*v[q] +((u[q])*u[q] - (v[q])*v[q])*N[q] +2*u[q]*(v[q])*Pdag[q]\
            +2*(u[q])*v[q]*P[q]

N_k = 2*(v[r])*v[r] +((u[r])*u[r] - (v[r])*v[r])*N[r] +2*u[r]*(v[r])*Pdag[r]\
       +2*(u[r])*v[r]*P[r]

In [ ]:
p,q,r,s,i,j,k,l = dr.names.A_dumms[:8]
 
 
u, v = IndexedBase('u'), IndexedBase('v')
E0 = Symbol(r'G')
H120, H021 = IndexedBase('h120') , IndexedBase('h021')
H011, H110 = IndexedBase('h011'), IndexedBase('h110')
H100, H011 = IndexedBase('h100'), IndexedBase('h011')
H001, H020 = IndexedBase('h001'), IndexedBase('h020')
H010 = IndexedBase('h010') 
H000 = IndexedBase('h000')


In [5]:
H11, H20 , H02 = IndexedBase('H11'), IndexedBase('H20'), IndexedBase('H02')
H04, H40 = IndexedBase('H04'), IndexedBase('H40')
H22, Hb22 = IndexedBase('H22'), IndexedBase('Hb22')
H31, H13 = IndexedBase('H31'), IndexedBase('H13')
dr.set_symm(H04,Perm([1,0],IDENT))
dr.set_symm(H40,Perm([1,0],IDENT)) 

In [6]:
expr = H11[p]*N[p] + H02[p]*Pdag[p] +H20[p]*P[p]+H22[p,q]*N[p]*N[q]+Hb22[p,q]*Pdag[p]*P[q]\
    +H40[p,q]*P[p]*P[q]+H04[p,q]*Pdag[p]*Pdag[q]+H13[p,q]*Pdag[p]*N[q]+H31[p,q]*N[p]*P[q]
H_N = dr.einst(expr).simplify().merge()

In [7]:
H_N.display()

<IPython.core.display.Math object>

In [8]:
t1, t2, t3,t4 = IndexedBase('t1'), IndexedBase('t2'), IndexedBase('t3'), IndexedBase('t4')

z1, z2 = IndexedBase('z1'), IndexedBase('z2')
z3, z4 = IndexedBase('z3'), IndexedBase('z4')
dr.set_symm(t2, Perm([1, 0],IDENT), valence=2)
dr.set_symm(t3, Perm([1,0,2],IDENT), Perm([0,2,1],IDENT), valence=3) 
dr.set_symm(z2, Perm([1, 0],IDENT), valence=2)
dr.set_symm(z3, Perm([1,0,2],IDENT), Perm([0,2,1],IDENT), valence=3)
dr.set_symm(t4,Perm([1,0,2,3],IDENT),Perm([0,1,3,2],IDENT),Perm([0,2,1,3],IDENT),valence=4)
dr.set_symm(z4,Perm([1,0,2,3],IDENT),Perm([0,1,3,2],IDENT),Perm([0,2,1,3],IDENT),valence=4)

Z1 = dr.einst(z1[p]*P[p])
Z2 = dr.einst(Rational(1,2)*z2[p,q]*P[p]*P[q])
Z3 = dr.einst(Rational(1,6)*z3[p,q,r]*P[p]*P[q]*P[r])
Z4 = dr.einst(Rational(1,24)*z4[p,q,r,s]*P[p]*P[q]*P[r]*P[s])

T1 = dr.einst(t1[p] * Pdag[p])
T2 = dr.einst(Rational(1, 2) * t2[p,q] *Pdag[p]*Pdag[q])
T3 = dr.einst(Rational(1,6)*t3[p,q,r]*Pdag[p]*Pdag[q]*Pdag[r])
T4 = dr.einst(Rational(1,24)*t4[p,q,r,s]*Pdag[p]*Pdag[q]*Pdag[r]*Pdag[s])

Z =  Z1+Z2 #+Z3+ Z4
cluster = T1+ T2 #+T3+ T4
Z.display()
#cluster.display()

<IPython.core.display.Math object>

In [9]:
zero_term = [(H40[p,p],0),(H04[p,p],0),(t2[p,p],0),(t4[p,p,p,p],0),(t3[p,p,p],0),(t3[p,p,q],0),(t3[p,q,p],0),(z3[p,p,p],0),(z3[q,p,p],0),(z3[p,p,q],0),(z2[p,p],0),(z4[p,p,p,p],0),
    (t3[q,p,p],0),(t3[p,q,p],0),(t4[p,p,r,s],0),(t4[r,p,p,s],0),(t4[r,s,p,p],0),(t4[p,r,p,s],0),(t4[p,r,s,p],0),(t4[r,p,s,p],0),
            (z4[p,p,r,s],0),(z4[r,p,p,s],0),(z4[r,s,p,p],0),(z4[p,r,p,s],0),(z4[p,r,s,p],0),(z4[r,p,s,p],0)]

In [10]:
t0 = time.time()
Hbar = H_N
curr = H_N
fact = 1
for n in range(6):                              # up to 4-fold for 2-body H
    comm = (curr | cluster).simplify()          # ad_T^{n+1}(H_N)
    fact *= (n + 1)                             # 1!,2!,3!,4!
    Hbar += comm / fact                         # add 1/(n+1)! term
    curr = comm    
print("Hbar evaluation done",time.time()-t0)

[Stage 193:================================================>   (269 + 12) / 288]

Hbar evaluation done 15.18486213684082


In [11]:
def drop_terms_containing_P(expr):

    ''' Assuming expr is normal_ordered '''
    kept = expr.simplify().filter(lambda term: all(v.label != 'P' for v in term.vecs))   #v.base
    #return Tensor(dr, kept).simplify().merge()
    return kept

def keep_exactly_k_Pdag(expr, k ,pdag_label='P^\\dagger'):

    """
    Keep only terms with exactly k creators P^\\dagger in the operator string.
    Assumes expr has already had all P (annihilators) filtered out and is normal ordered.
    """
    def nPdag(term):
        return sum(1 for v in term.vecs if v.label == pdag_label)

    kept = expr.simplify().filter(lambda term: nPdag(term) == k)
    #return Tensor(dr, kept).simplify().merge()
    return kept
    

In [12]:
Hbar_noP = drop_terms_containing_P(Hbar) # for energy and Z equations
Hbar_noP.n_terms

129

In [ ]:
E_eqn  = Hbar_noP.simplify().eval_vev().simplify()
E_eqn = E_eqn.subst_all(zero_term)
#print(E_eqn.latex())
E_eqn.display()

                                                                                



In [13]:
Hbar_T1 = keep_exactly_k_Pdag(Hbar_noP,1)
Hbar_T1.n_terms

36

In [14]:
t0=time.time()
s1_eqn_CCSD = (P[p] * Hbar_T1).simplify().eval_vev().simplify()
print('evaluation done',time.time()-t0)

evaluation done 3.519557237625122


In [15]:
s1_eqn_CCSD = s1_eqn_CCSD.subst_all(zero_term)
print(s1_eqn_CCSD.latex())

\sum_{q \in A} H^{(20)}_{q}  t^{(2)}_{q,p}   + \sum_{q \in A} t^{(1)}_{q}  HT^{(22)}_{p,q}   + \sum_{q \in A} 2 H^{(31)}_{p,q}  t^{(2)}_{q,p}  - \sum_{q \in A} 4 t^{(1)}_{p}  H^{(40)}_{q,p}  t^{(2)}_{q,p}  - \sum_{q \in A} 2 t^{(1)}_{p}^{2}  t^{(1)}_{q}  H^{(40)}_{q,p}   + \sum_{q \in A} 2 t^{(1)}_{p}  t^{(1)}_{q}  H^{(31)}_{p,q}  - t^{(1)}_{p}^{2}  H^{(20)}_{p}  - 2 t^{(1)}_{p}^{2}  H^{(31)}_{p,p}   + 2 H^{(11)}_{p}  t^{(1)}_{p}   + 4 t^{(1)}_{p}  H^{(22)}_{p,p}   + H^{(02)}_{p}   + \sum_{q \in A} \sum_{r \in A} 2 t^{(1)}_{q}  H^{(40)}_{q,r}  t^{(2)}_{r,p} 


In [13]:
Hbar_T2 = keep_exactly_k_Pdag(Hbar_noP,2)
Hbar_T2.n_terms

42

In [14]:
t0 = time.time()
s2_eqn_CCSD = (P[p]*P[q] * Hbar_T2).simplify().eval_vev().simplify()
s2_eqn_CCSD = s2_eqn_CCSD.subst_all(zero_term)
print('evaluation done ', time.time() - t0)

evaluation done  20.00342297554016


In [ ]:
s2_eqn_CCSD.n_terms

In [19]:
print(s2_eqn_CCSD.merge().latex())


- \sum_{r \in A} \sum_{s \in A} \left(2 \delta_{p q} H^{(40)}_{r,s} t^{(2)}_{r,p} t^{(2)}_{s,p} - 2 H^{(40)}_{r,s} t^{(2)}_{r,p} t^{(2)}_{s,q}\right)  - \sum_{r \in A} \left(4 \delta_{p q} H^{(31)}_{p,r} t^{(1)}_{p} t^{(2)}_{r,p} - 4 \delta_{p q} H^{(40)}_{r,p} t^{(1)}_{p}^{2} t^{(2)}_{r,p} + 2 \delta_{p q} Hb^{(22)}_{p,r} t^{(2)}_{r,p} - 2 H^{(31)}_{p,r} t^{(1)}_{p} t^{(2)}_{r,q} - 2 H^{(31)}_{p,r} t^{(1)}_{r} t^{(2)}_{p,q} - 2 H^{(31)}_{q,r} t^{(1)}_{q} t^{(2)}_{r,p} - 2 H^{(31)}_{q,r} t^{(1)}_{r} t^{(2)}_{p,q} + 2 H^{(40)}_{r,p} t^{(1)}_{p}^{2} t^{(2)}_{r,q} + 4 H^{(40)}_{r,p} t^{(1)}_{p} t^{(1)}_{r} t^{(2)}_{p,q} + 4 H^{(40)}_{r,p} t^{(2)}_{p,q} t^{(2)}_{r,p} + 2 H^{(40)}_{r,q} t^{(1)}_{q}^{2} t^{(2)}_{r,p} + 4 H^{(40)}_{r,q} t^{(1)}_{q} t^{(1)}_{r} t^{(2)}_{p,q} + 4 H^{(40)}_{r,q} t^{(2)}_{p,q} t^{(2)}_{r,q} - Hb^{(22)}_{p,r} t^{(2)}_{r,q} - Hb^{(22)}_{q,r} t^{(2)}_{r,p}\right)  - \left(4 \delta_{p q} H^{(13)}_{p,p} t^{(1)}_{p} + 8 \delta_{p q} H^{(22)}_{p,p} t^{(1)}_{p}^{2} - 4 \

In [24]:
def drop_delta_terms_str(tens: Tensor) -> Tensor:
    # Remove any term whose printed form contains 'KroneckerDelta('
    filt = tens.simplify().filter(lambda term: 'KroneckerDelta' not in str(term))
     
    #return Tensor(tens._drudge, filt).simplify()
    return filt

 

In [ ]:
s2_eqn_CCSD_.n_terms

In [ ]:
'''Defining CC Lagrangian
L = <0|(1+Z) bar{H}|0> and dL/dt = 0 is the Z equations. dL/dt = <0|(1+Z)[bar{H},Q_{mu}]|0>
'''
t0 = time.time()
# 1) Build the commutator object
C = (Hbar | Pdag[p])             

# 2) Simplify the commutator itself (usually shrinks a lot)
C = C.simplify()

# 3) Use linearity: <0|(1+Z) C|0> = <0|C|0> + <0|Z C|0>
#term0 = C.eval_vev().simplify()

C0 = keep_exactly_k_Pdag(C, 0)
C1 = keep_exactly_k_Pdag(C, 1)
C2 = keep_exactly_k_Pdag(C, 2)
C3 = keep_exactly_k_Pdag(C, 3)
C4 = keep_exactly_k_Pdag(C, 4)
term0 = C0.eval_vev().simplify()

term1 = (Z1 * C1).simplify().eval_vev().simplify()
term2 = (Z2 * C2).simplify().eval_vev().simplify()
term3 = (Z3 * C3).simplify().eval_vev().simplify()
term4 = (Z4 * C4).simplify().eval_vev().simplify()

dL_dt_1 = (term0 + term1 + term2 + term3 + term4).simplify().merge()

#termZ4 = (Z4 * C).simplify().eval_vev().simplify() 
#termZ3 = (Z3 * C).simplify().eval_vev().simplify() 
#termZ2 = (Z2 * C).simplify().eval_vev().simplify() 
#termZ1 = (Z1 * C).simplify().eval_vev().simplify() 
#termZ = (Z * C).normal_order().eval_vev().simplify() 
#dL_dt_1 = (term0 + termZ).simplify().merge() 
 
#dL_dt_1 = ((1+Z)*(Hbar|Pdag[p])).simplify().eval_vev().simplify().merge()
print("finish time: ",time.time()-t0 )

In [ ]:
termZ3.simplify().display()

In [ ]:
dL_dt_1 = dL_dt_1.subst_all(zero_term)
print(dL_dt_1.n_terms)

In [ ]:
print(dL_dt_1.latex())

In [ ]:
t0 = time.time()
# 1) Build the commutator object
#Z2 equations
C = (Hbar | Pdag[p]*Pdag[q])
C = C.normal_order().simplify().merge()
print("finish time: ",time.time()-t0 )

In [ ]:
C.n_terms

In [ ]:
C_noP = drop_terms_containing_P(C)
C_noP.n_terms

In [ ]:
t0 = time.time()
C0 = keep_exactly_k_Pdag(C_noP, 0)
C1 = keep_exactly_k_Pdag(C_noP, 1)
C2 = keep_exactly_k_Pdag(C_noP, 2)
C3 = keep_exactly_k_Pdag(C_noP, 3)
C4 = keep_exactly_k_Pdag(C_noP, 4)
term0 = C0.eval_vev().simplify()
term1 = (Z1 * C1).simplify().eval_vev()
term2 = (Z2 * C2).simplify().eval_vev()
term3 = (Z3 * C3).simplify().eval_vev()
#term4 = (Z4 * C4).simplify().eval_vev()
print("finish time: ",time.time()-t0 )

In [ ]:
'''Defining CC Lagrangian
L = <0|(1+Z) bar{H}|0> and dL/dt = 0 is the Z equations. dL/dt = <0|(1+Z)[bar{H},Q_{mu}]|0>
'''
t0 = time.time()
# 1) Build the commutator object
#Z2 equations
C = (Hbar | Pdag[p]*Pdag[q])             

# 2) Simplify the commutator itself (usually shrinks a lot)
C = C.simplify()
C_noP = drop_terms_containing_P(C)
# 3) Use linearity: <0|(1+Z) C|0> = <0|C|0> + <0|Z C|0>
#term0 = C.eval_vev().simplify()

C0 = keep_exactly_k_Pdag(C_noP, 0)
C1 = keep_exactly_k_Pdag(C_noP, 1)
C2 = keep_exactly_k_Pdag(C_noP, 2)
C3 = keep_exactly_k_Pdag(C_noP, 3)
C4 = keep_exactly_k_Pdag(C_noP, 4)
term0 = C0.eval_vev().simplify()

term1 = (Z1 * C1).simplify().eval_vev()
term2 = (Z2 * C2).simplify().eval_vev()
term3 = (Z3 * C3).simplify().eval_vev()
term4 = (Z4 * C4).simplify().eval_vev()

dL_dt_2 = (term0 + term1 + term2 + term3 + term4).simplify().merge()

#termZ4 = (Z4 * C).simplify().eval_vev().simplify() 
#termZ3 = (Z3 * C).simplify().eval_vev().simplify() 
#termZ2 = (Z2 * C).simplify().eval_vev().simplify() 
#termZ1 = (Z1 * C).simplify().eval_vev().simplify() 
#termZ = (Z * C).normal_order().eval_vev().simplify() 
#dL_dt_1 = (term0 + termZ).simplify().merge() 
 
#dL_dt_1 = ((1+Z)*(Hbar|Pdag[p])).simplify().eval_vev().simplify().merge()
print("finish time: ",time.time()-t0 )

In [ ]:
dL_dt_2_del = drop_delta_terms_str(dL_dt_2)
dL_dt_2_del.display()

In [ ]:
X = (term0+term1 + term2 + term3+ term4).simplify()
X = drop_delta_terms_str(X)
X.display()

In [ ]:
term1_del = drop_delta_terms_str(term1)
term2_del = drop_delta_terms_str(term2)
term3_del = drop_delta_terms_str(term3)
term4_del = drop_delta_terms_str(term4)


In [ ]:
dL_dt_2_exp = (term0 + term1_del + term2_del + term3_del + term4_del).simplify()
dL_dt_2_exp =  dL_dt_2_exp.subst_all(zero_term)

In [ ]:
print(dL_dt_2_exp.latex())

In [ ]:
sum2 = term1_del + term2_del + term3_del + term4_del
sum2_z = sum2.subst_all(zero_term).simplify().merge()
print(sum2_z.latex())

In [ ]:
#dL_dt_2 = (1+Z)*(Hbar|Pdag[p]*Pdag[q]).simplify().eval_vev().simplify()
# 1) Build the commutator object
'''t0 = time.time()
C = (Hbar | (Pdag[p]*Pdag[q]))             

# 2) Simplify the commutator itself (usually shrinks a lot)
C = C.simplify()

# 3) Use linearity: <0|(1+Z) C|0> = <0|C|0> + <0|Z C|0>
term0 = C.eval_vev().simplify()
termZ = (Z * C).simplify().eval_vev().simplify()   # crucial: eval_vev BEFORE simplify of the product

dL_dt_2 = (term0 + termZ).simplify()
print("finish", time.time()-t0)
dL_dt_2 = dL_dt_2.subst_all(zero_term)
print(dL_dt_2.latex())
'''

In [ ]:
 
L1 = IndexedBase('L1')
L2 = IndexedBase('L2')

expr1 = dr.define(L1[p],dL_dt_1)
 
#print("LHS:", expr1.lhs)
#print("RHS type:", type(expr1.rhs))
#print("RHS:", expr1.rhs)
expr2 = dr.define(L2[p,q],dL_dt_2)
 
 
 
eval_equ0 = optimize(
    [expr1,expr2],
    interm_fmt='tau{}',drop_cutoff=2
)
print("Original cost:", get_flop_cost(eval_equ0))

fort = FortranPrinter()
print("Optimized cost:", get_flop_cost(eval_equ0))
 
code = fort.doprint(eval_equ0)



with open("SFS_BCS_CCSD_Z.f90", "w") as fp:
    fp.write(code)

print('Done')

In [15]:
import time
from drudge import Tensor
from gristmill import optimize, get_flop_cost
from gristmill.generate import FortranPrinter
from sympy import IndexedBase
from sympy.printing.fortran import FCodePrinter

# Patch SymPy's Fortran printer to understand KroneckerDelta
def _print_KroneckerDelta(self, expr):
    i, j = expr.args
    return f"merge(1, 0, {self._print(i)} == {self._print(j)})"

FCodePrinter._print_KroneckerDelta = _print_KroneckerDelta

t0 = time.time()
Res2 = IndexedBase('R2')

expr1 = dr.define(Res2[p, q], s2_eqn_CCSD)

eval_equ0 = optimize(
    [expr1],
    interm_fmt='tau{}',
    drop_cutoff=2
)

print(type(eval_equ0))
print("Optimized cost:", get_flop_cost(eval_equ0))

fort = FortranPrinter()
evals_list = fort.doprint(eval_equ0)

with open("test.f90", "w") as fp:
    fp.write(evals_list)

print("Printed!", time.time() - t0)

/Users/swarnamoyghosh/anaconda3/envs/drudge_fix/lib/python3.9/site-packages/gristmill/optimize.py:1984: UserWarning: Internal deficiency: summation intermediate may not be fully canonicalized
  warnings.warn(


<class 'list'>
Optimized cost: 8*na**3 + 54*na**2 + 16*na
Printed! 8.54363226890564


In [17]:
#Lmax = symbols('Lmax', integer=True, positive=True)
#L = Range('L', 1, Lmax)  # symbolic chain 1..Lmax (boundary handling is symbolic)
#dr.set_dumms(L, (p,q))
#expr = Rational(1,4)*(Pdag[p] + P[p]) - Rational(1,2)*(N[p-1]-1)*(Pdag[p] + P[p])*(N[p+1] - 1) +Rational(1,4)*G*(N[p-1] - 1)*(N[p+1] - 1)
#expr =  Rational(1,4)*(N[p]*N[p+r] - N[p] - N[p+r] +1)
Corr =  Rational(1,4)*(N_i*N_j - N_i - N_j +1) + (P_i_dag*P_j + P_j_dag*P_i)/2  #Corr is the Sp.Sq correlation function
#Corr =  Rational(1/4)*(N_i*N_j - N_i - N_j +1) + (Pdag[p]*P[q] + Pdag[q]*P[p])/2
  

In [18]:
dr.sum(Corr).simplify().merge().map2amps(factor).display()

<IPython.core.display.Math object>

In [19]:
t0 =time.time()
SpSqbar = Corr
curr = Corr
fact = 1
for n in range(6):                              # up to 4-fold for 2-body H
    comm = (curr | cluster).simplify()          # ad_T^{n+1}(H_N)
    fact *= (n + 1)                             # 1!,2!,3!,4!
    SpSqbar += comm / fact                         # add 1/(n+1)! term
    curr = comm    
print("SpSqbar evaluation done ", time.time()-t0)


[Stage 3127:=================================================> (278 + 10) / 288]

SpSqbar evaluation done  26.03963279724121


In [20]:
t0 = time.time()
    # you will set one(i)=1 in code
Ex_SpSq =  ((1+Z)*(SpSqbar)).simplify().eval_vev().simplify()
print("finished ", time.time()-t0)

                                                                                



finished  397.1325876712799


In [21]:
Ex_SpSq  = Ex_SpSq.subst_all(zero_term)
#Ex_SpSq = drop_delta_terms_str(Ex_SpSq).simplify()

print(Ex_SpSq.latex())

- \sum_{r \in A} \sum_{s \in A}  \frac{1}{2} u_{p}^{2}  v_{q}^{2}  t^{(2)}_{r,p}  t^{(2)}_{s,q}  z^{(2)}_{r,s}  - \sum_{r \in A} \sum_{s \in A}  \frac{1}{2} u_{q}^{2}  v_{p}^{2}  t^{(2)}_{r,p}  t^{(2)}_{s,q}  z^{(2)}_{r,s}   + \sum_{r \in A} \sum_{s \in A} u_{p}  u_{q}  v_{p}  v_{q}  t^{(2)}_{r,p}  t^{(2)}_{s,q}  z^{(2)}_{r,s}   + \sum_{r \in A} \frac{1}{2} v_{p}^{2}  t^{(2)}_{r,p}  z^{(2)}_{r,p}   + \sum_{r \in A} \frac{1}{2} v_{q}^{2}  t^{(2)}_{r,q}  z^{(2)}_{r,q}  - \sum_{r \in A}  \frac{1}{2} u_{p}^{2}  t^{(2)}_{r,p}  z^{(2)}_{r,p}  - \sum_{r \in A}  \frac{1}{2} u_{q}^{2}  t^{(2)}_{r,q}  z^{(2)}_{r,q}   + \sum_{r \in A} u_{p}^{2}  v_{q}^{2}  t^{(2)}_{r,p}  z^{(2)}_{r,p}   + \sum_{r \in A} \delta_{p q} u_{p}^{4}  t^{(2)}_{r,p}  z^{(2)}_{r,p}   + \sum_{r \in A} u_{q}^{2}  v_{p}^{2}  t^{(2)}_{r,q}  z^{(2)}_{r,q}   + \sum_{r \in A} \frac{1}{2} u_{p}^{2}  u_{q}^{2}  t^{(2)}_{r,p}  z^{(2)}_{r,q}   + \sum_{r \in A} \frac{1}{2} u_{p}^{2}  u_{q}^{2}  t^{(2)}_{r,q}  z^{(2)}_{r,p}   + \sum_{r

In [19]:
Ex_SpSq.n_terms

206

In [20]:
Ex_SpSq.display()

<IPython.core.display.Math object>

In [ ]:
from drudge import Tensor
from gristmill import  optimize, get_flop_cost
from gristmill.generate import FortranPrinter
from sympy import IndexedBase

t0 = time.time()
SpSq = IndexedBase('SpSq')
 
expr1 = dr.define(SpSq[p,q],Ex_SpSq) 
 
 
orig_eqn = [expr1]
print("Original cost:", get_flop_cost(orig_eqn))

eval_equ0 = optimize(
    [expr1],
    interm_fmt='tau{}',drop_cutoff=2
)
print(type(eval_equ0))
 
fort = FortranPrinter()
print("Optimized cost:", get_flop_cost(eval_equ0))

# eval_equ0 is an iterable of TensorDef objects
decls, evals_list = fort.print_decl_eval(eval_equ0, decl_type='real', explicit_bounds=False)

code = "\n".join(decls) + "\n\n" + "\n\n".join(evals_list)

with open("SFS_BCS_CCSDSpSq.f90", "w") as fp:
    fp.write(code)

print("Printed!", time.time()-t0)


Original cost: 17*na**4 + 390*na**3 + 705*na**2


In [23]:
from drudge import Tensor
from gristmill import  optimize, get_flop_cost
from gristmill.generate import FortranPrinter
from sympy import IndexedBase

t0 = time.time()
SpSq = IndexedBase('SpSq')
 
expr1 = dr.define(SpSq[p,q],Ex_SpSq) 
 
 

eval_equ0 = optimize(
    [expr1],
    interm_fmt='tau{}',drop_cutoff=2
)
print(type(eval_equ0))
print("Original cost:", get_flop_cost(eval_equ0))

fort = FortranPrinter()
print("Optimized cost:", get_flop_cost(eval_equ0))

# eval_equ0 is an iterable of TensorDef objects
evals_list = fort.doprint(eval_equ0)

with open("SFS_BCS_CCSDSpSq_test.f90", "w") as fp:
    fp.write(evals_list)

print("Printed!", time.time()-t0)

NameError: name 'Ex_SpSq' is not defined

In [ ]:
import time
from drudge import Tensor
from gristmill import optimize, get_flop_cost
from gristmill.generate import FortranPrinter
from sympy import IndexedBase
from sympy.printing.fortran import FCodePrinter

# Patch SymPy's Fortran printer to understand KroneckerDelta
def _print_KroneckerDelta(self, expr):
    i, j = expr.args
    return f"merge(1, 0, {self._print(i)} == {self._print(j)})"

FCodePrinter._print_KroneckerDelta = _print_KroneckerDelta

t0 = time.time()
SpSq = IndexedBase('SpSq')

expr1 = dr.define(SpSq[p, q], Ex_SpSq)

eval_equ0 = optimize(
    [expr1],
    interm_fmt='tau{}',
    drop_cutoff=2
)

print(type(eval_equ0))
print("Optimized cost:", get_flop_cost(eval_equ0))

fort = FortranPrinter()
evals_list = fort.doprint(eval_equ0)

with open("SFS_BCS_CCSDSpSq_test.f90", "w") as fp:
    fp.write(evals_list)

print("Printed!", time.time() - t0)

In [16]:
import time
from drudge import Tensor
from gristmill import optimize, get_flop_cost
from gristmill.generate import FortranPrinter
from sympy import IndexedBase
from sympy.printing.fortran import FCodePrinter

# Patch SymPy Fortran printer: KroneckerDelta(i,j) -> deltaf(i,j)
def _print_KroneckerDelta(self, expr):
    i, j = expr.args
    return f"deltaf({self._print(i)}, {self._print(j)})"

FCodePrinter._print_KroneckerDelta = _print_KroneckerDelta

t0 = time.time()
SpSq = IndexedBase('SpSq')

expr1 = dr.define(SpSq[p, q], Ex_SpSq)

eval_equ0 = optimize(
    [expr1],
    interm_fmt='tau{}',
    drop_cutoff=2
)

print(type(eval_equ0))
print("Optimized cost:", get_flop_cost(eval_equ0))

fort = FortranPrinter()
evals_list = fort.doprint(eval_equ0)

with open("SFS_BCS_CCSDSpSq_test.f90", "w") as fp:
    fp.write(evals_list)

print("Printed!", time.time() - t0)

NameError: name 'Ex_SpSq' is not defined

In [17]:
import time
from drudge import Tensor
from gristmill import optimize, get_flop_cost
from gristmill.generate import FortranPrinter
from sympy import IndexedBase
from sympy.printing.fortran import FCodePrinter

# Patch SymPy's Fortran printer:
# KroneckerDelta(i,j) -> deltaf(i,j)
def _print_KroneckerDelta(self, expr):
    i, j = expr.args
    return f"deltaf({self._print(i)}, {self._print(j)})"

FCodePrinter._print_KroneckerDelta = _print_KroneckerDelta

t0 = time.time()
Res2 = IndexedBase('R2')

expr1 = dr.define(Res2[p, q], s2_eqn_CCSD)

eval_equ0 = optimize(
    [expr1],
    interm_fmt='tau{}',
    drop_cutoff=2
)

print(type(eval_equ0))
print("Optimized cost:", get_flop_cost(eval_equ0))

fort = FortranPrinter()
evals_list = fort.doprint(eval_equ0)

delta_func = """
pure double precision function deltaf(p, q)
  implicit none
  integer, intent(in) :: p, q

  if (p == q) then
    deltaf = 1.0d0
  else
    deltaf = 0.0d0
  end if

end function deltaf
"""

with open("test.f90", "w") as fp:
    fp.write(delta_func)
    fp.write("\n\n")
    fp.write(evals_list)

print("Printed!", time.time() - t0)

<class 'list'>
Optimized cost: 8*na**3 + 54*na**2 + 16*na
Printed! 8.193713903427124
